# CHRONOS | SPHEREx DR1 real-data audit, calibration matching, alignment and recovery benchmark

**Kaggle Notebook edition.** Attach `SPHEREx_DR1_Research_Bundle.zip` as a Kaggle Dataset using **Add Input**. Turn **Notebook options → Internet = On** so this notebook can query IRSA/Gaia and install missing packages.

This notebook audits the Level-2 MEF bytes, obtains decoded-flag IRSA cutouts, discovers detector-matched calibration files, compares calibration provenance, measures Gaia/ePSF alignment when data permit, and runs real-image injection/recovery tests.

**Kaggle setup:** upload the ZIP as a **Private Kaggle Dataset**, attach it via Add Input, enable Internet, and run from the top. Files are read from `/kaggle/input` and all working/results files go to `/kaggle/working/chronos_spherex_work`.

**Scientific boundaries:** the embedded WCS-WAVE table is for approximate visualization; science wavelength work needs standalone `CWAVE`/`CBAND`. The Level-2 IMAGE is already in MJy/sr: **do not apply the Absolute Gain Matrix again**. Detector alignment and completeness are only measured if the corresponding real data and tests succeed. A successful run is not itself validation.

Official references: [SPHEREx Data Products](https://caltech-ipac.github.io/spherex-archive-documentation/spherex-data-products/), [Data Access](https://caltech-ipac.github.io/spherex-archive-documentation/spherex-data-access/), [IRSA ePSF tutorial](https://caltech-ipac.github.io/irsa-tutorials/spherex-psf/).


In [ ]:
# Kaggle Internet must be ON for package installation.
%pip -q install astropy photutils astroquery pyvo reproject requests pandas scipy matplotlib seaborn tqdm beautifulsoup4


In [ ]:
from pathlib import Path
import os, re, io, json, csv, time, math, hashlib, shutil, zipfile, warnings, textwrap
from urllib.parse import urljoin, urlparse
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u
from astropy.utils.data import conf
from astroquery.ipac.irsa import Irsa
from astroquery.gaia import Gaia
from photutils.psf import ImagePSF
from scipy.optimize import least_squares
from scipy.ndimage import maximum_filter
from scipy.interpolate import RegularGridInterpolator
from bs4 import BeautifulSoup

conf.remote_timeout = 180
WORK = Path('/kaggle/working/chronos_spherex_work')
WORK.mkdir(parents=True, exist_ok=True)
RAW = WORK/'raw'; RAW.mkdir(exist_ok=True)
CAL = WORK/'calibrations'; CAL.mkdir(exist_ok=True)
OUT = WORK/'results'; OUT.mkdir(exist_ok=True)
KAGGLE_INPUT = Path('/kaggle/input')
print('Work directory:', WORK)
print('Kaggle input root exists:', KAGGLE_INPUT.exists())


## 1. Locate the SPHEREx bundle in Kaggle Inputs

Kaggle does not use the Colab upload picker. Attach the ZIP as a Dataset via **Add Input**. The next cell searches `/kaggle/input` for the original filename. If it is already extracted in the Dataset, the cell can use the manifest and MEF directly.

If you uploaded it under another name, set `ZIP_PATH_OVERRIDE` to the full path shown in Kaggle's Input browser.


In [ ]:
# Kaggle Input files are read-only. Extract/copy working products to /kaggle/working.
ZIP_PATH_OVERRIDE = None  # e.g. '/kaggle/input/my-spherex-data/SPHEREx_DR1_Research_Bundle.zip'
BUNDLE = WORK/'bundle'
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True, exist_ok=True)

zip_path = Path(ZIP_PATH_OVERRIDE) if ZIP_PATH_OVERRIDE else None
if zip_path is not None and not zip_path.exists(): raise FileNotFoundError(f'ZIP_PATH_OVERRIDE does not exist: {zip_path}')
if zip_path is None:
    if not KAGGLE_INPUT.exists(): raise FileNotFoundError('No /kaggle/input. Attach your SPHEREx ZIP as a Kaggle Dataset via Add Input.')
    zip_candidates = list(KAGGLE_INPUT.rglob('SPHEREx_DR1_Research_Bundle.zip'))
    if not zip_candidates:
        for candidate in KAGGLE_INPUT.rglob('*.zip'):
            try:
                with zipfile.ZipFile(candidate) as zf:
                    names=zf.namelist()
                    if any(n.endswith('DATA_MANIFEST.txt') for n in names) and any(Path(n).name.startswith('level2_') and n.lower().endswith('.fits') for n in names): zip_candidates.append(candidate)
            except Exception: pass
    if zip_candidates: zip_path=sorted(zip_candidates,key=lambda q:(q.name!='SPHEREx_DR1_Research_Bundle.zip',str(q)))[0]

if zip_path is not None:
    ZIP_PATH=zip_path
    print('Bundle ZIP:',ZIP_PATH,'bytes:',ZIP_PATH.stat().st_size)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        for zi in zf.infolist(): print(f'{zi.filename}  {zi.file_size:,} bytes')
        zf.extractall(BUNDLE)
    manifest_files=list(BUNDLE.rglob('DATA_MANIFEST.txt')); mef_files=list(BUNDLE.rglob('level2_*.fits'))
    if not manifest_files or not mef_files: raise FileNotFoundError('ZIP must contain DATA_MANIFEST.txt and at least one level2_*.fits file')
    MANIFEST_PATH=manifest_files[0]; MEF_PATH=mef_files[0]
else:
    ZIP_PATH=None
    found=None
    for m in KAGGLE_INPUT.rglob('DATA_MANIFEST.txt'):
        candidates=list(m.parent.rglob('level2_*.fits'))
        if candidates: found=(m,m.parent,candidates[0]); break
    if found is None: raise FileNotFoundError('Could not find SPHEREx ZIP or a DATA_MANIFEST.txt plus level2_*.fits under /kaggle/input. Attach the original ZIP as a Kaggle Dataset.')
    MANIFEST_PATH,BUNDLE,MEF_PATH=found

print('Manifest:',MANIFEST_PATH)
print(MANIFEST_PATH.read_text(errors='replace'))
print('MEF:',MEF_PATH,'bytes:',MEF_PATH.stat().st_size)


## 2. Byte-level FITS audit and primary science arrays

This cell computes SHA-256 locally, validates the expected named extensions, prints all HDUs and summarizes the image, variance and zodiacal arrays. It also inspects how the `FLAGS` HDU is actually encoded instead of assuming it is an ordinary integer image.

In [ ]:
def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

MEF_SHA256 = sha256_file(MEF_PATH)
print('SHA-256:', MEF_SHA256)

with fits.open(MEF_PATH, memmap=True, lazy_load_hdus=False) as hdul:
    hdul.info()
    primary_header = hdul[0].header.copy()
    image_header = hdul['IMAGE'].header.copy()
    mef_names = [h.name for h in hdul]
    print('\nMEF version:', primary_header.get('VERSION'))
    for key in ['DETECTOR','OBSID','EXPIDN','DATE-OBS','DATE-AVG','DATE-END','XPOSURE','PSF_FWHM','L1DQAFLG','L2DQAFLG','DETCOORD','OMEGA_MEDIAN','OMEGA_STDDEV']:
        print(f'{key}: {image_header.get(key)}')
    for name in ['IMAGE','FLAGS','VARIANCE','ZODI','EPSF','WCS-WAVE']:
        h = hdul[name]
        print(f'\n{name}: type={type(h).__name__}, shape={getattr(h.data, "shape", None)}, dtype={getattr(h.data, "dtype", None)}')
        if isinstance(h, fits.BinTableHDU):
            print('columns:', h.columns.names)
            for col in h.columns:
                print(' ', col.name, col.format, col.dim)
    image = np.asarray(hdul['IMAGE'].data, dtype=np.float32)
    variance = np.asarray(hdul['VARIANCE'].data, dtype=np.float32)
    zodi = np.asarray(hdul['ZODI'].data, dtype=np.float32)
    flags_hdu_type = type(hdul['FLAGS']).__name__
    flags_hdu_shape = getattr(hdul['FLAGS'].data, 'shape', None)
    image_wcs = WCS(hdul['IMAGE'].header)

assert image.shape == variance.shape == zodi.shape, 'Science arrays have inconsistent dimensions'

def stats(a):
    v = np.asarray(a)[np.isfinite(a)]
    return {'shape': list(a.shape), 'n_finite': int(v.size), 'finite_fraction': float(v.size/a.size),
            'min': float(v.min()), 'median': float(np.median(v)), 'mean': float(np.mean(v)),
            'std': float(np.std(v)), 'p99': float(np.percentile(v,99)), 'p999': float(np.percentile(v,99.9)), 'max': float(v.max())}

mef_audit = {
    'file': MEF_PATH.name, 'size_bytes': MEF_PATH.stat().st_size, 'sha256': MEF_SHA256,
    'hdu_names': mef_names, 'flags_hdu_type': flags_hdu_type, 'flags_hdu_shape': flags_hdu_shape,
    'primary_version': primary_header.get('VERSION'),
    'detector': image_header.get('DETECTOR'), 'obsid': image_header.get('OBSID'),
    'date_obs': image_header.get('DATE-OBS'), 'detcoord': image_header.get('DETCOORD'),
    'image': stats(image), 'variance': stats(variance), 'zodi': stats(zodi)
}
(OUT/'mef_byte_audit.json').write_text(json.dumps(mef_audit, indent=2))
print(json.dumps(mef_audit, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, arr, title in zip(axes, [image, variance, zodi], ['IMAGE (MJy/sr)', 'VARIANCE ((MJy/sr)^2)', 'ZODI (MJy/sr)']):
    lo, hi = np.nanpercentile(arr, [1, 99])
    im = ax.imshow(arr, origin='lower', vmin=lo, vmax=hi, cmap='magma')
    ax.set_title(title); ax.set_xlabel('x pixel'); ax.set_ylabel('y pixel')
    plt.colorbar(im, ax=ax, shrink=.75)
plt.tight_layout(); plt.savefig(OUT/'mef_planes_quicklook.png', dpi=160); plt.show()

## 3. Embedded ePSF integrity and approximate WCS-WAVE cross-check

IRSA documents the ePSF as an oversampled grid of spatial zones. It must be rendered at a source's sub-pixel phase and normalized after evaluation on native detector pixel centres. Do not treat a 33×33 oversampled array as a 33×33 native-pixel PSF.

In [ ]:
with fits.open(MEF_PATH, memmap=True, lazy_load_hdus=False) as hdul:
    epsf_hdu = hdul['EPSF']
    epsf_tab = Table(epsf_hdu.data)
    epsf_header = epsf_hdu.header.copy()
    wave_tab = hdul['WCS-WAVE'].data[0]
    wave_header = hdul['WCS-WAVE'].header.copy()

print('ePSF columns:', epsf_tab.colnames)
print('ePSF DETECTOR / DETCOORD:', epsf_header.get('DETECTOR'), epsf_header.get('DETCOORD'))
print('OVSMPX/Y:', epsf_header.get('OVSMPX'), epsf_header.get('OVSMPY'))
print('ePSF source:', [x for x in epsf_header.get('HISTORY', [])] if False else 'see HISTORY in the HDU header')

for required in ['XCENTER','YCENTER','BINX','BINY','EPSF','CWAVE','CBAND']:
    if required not in epsf_tab.colnames:
        raise KeyError(f'ePSF table lacks expected field {required}; actual fields: {epsf_tab.colnames}')

psf_arrays = np.asarray([np.asarray(x).reshape(33,33) for x in epsf_tab['EPSF']], dtype=float)
psf_sums = psf_arrays.sum(axis=(1,2))
psf_finite = np.isfinite(psf_arrays).all(axis=(1,2))
ex = np.asarray(epsf_tab['XCENTER'], float); ey = np.asarray(epsf_tab['YCENTER'], float)
print('zones:', len(epsf_tab), 'grid:', len(np.unique(epsf_tab['BINX'])), 'x', len(np.unique(epsf_tab['BINY'])), 'y')
print('ePSF sum min/median/max:', float(psf_sums.min()), float(np.median(psf_sums)), float(psf_sums.max()))
print('all ePSF arrays finite:', bool(psf_finite.all()))
print('zone center extents:', [float(ex.min()),float(ex.max())], [float(ey.min()),float(ey.max())])

xctrl = np.asarray(wave_tab['X'], float).ravel()
yctrl = np.asarray(wave_tab['Y'], float).ravel()
values = np.asarray(wave_tab['VALUES'], float).reshape(len(yctrl),len(xctrl),2)
wave_interp = RegularGridInterpolator((yctrl,xctrl), values[:,:,0], bounds_error=False, fill_value=np.nan)
band_interp = RegularGridInterpolator((yctrl,xctrl), values[:,:,1], bounds_error=False, fill_value=np.nan)
# Control points in WCS-WAVE are FITS 1-based; EPSF zone centers are 0-based.
pts = np.column_stack([ey+1, ex+1])
wave_approx = wave_interp(pts); band_approx = band_interp(pts)
dwave_nm = (np.asarray(epsf_tab['CWAVE'],float)-wave_approx)*1000
dband_nm = (np.asarray(epsf_tab['CBAND'],float)-band_approx)*1000

In [ ]:
# Report the internal ePSF-vs-WCS-WAVE consistency only. It is NOT an independent calibration alignment test.
wave_check = {
    'n_zones': int(len(epsf_tab)),
    'wcs_wave_approx_um_range': [float(np.nanmin(wave_approx)), float(np.nanmax(wave_approx))],
    'epsf_cwave_minus_wcswave_nm_median': float(np.nanmedian(dwave_nm)),
    'epsf_cwave_minus_wcswave_nm_p95_abs': float(np.nanpercentile(np.abs(dwave_nm),95)),
    'epsf_cwave_minus_wcswave_nm_max_abs': float(np.nanmax(np.abs(dwave_nm))),
    'epsf_cband_minus_wcswave_nm_median': float(np.nanmedian(dband_nm)),
    'epsf_cband_minus_wcswave_nm_p95_abs': float(np.nanpercentile(np.abs(dband_nm),95)),
    'epsf_cband_minus_wcswave_nm_max_abs': float(np.nanmax(np.abs(dband_nm))),
    'qualification': 'embedded WCS-WAVE is approximate; standalone CWAVE/CBAND required for science wavelength validation'
}
(OUT/'embedded_wcs_wave_crosscheck.json').write_text(json.dumps(wave_check, indent=2))
print(json.dumps(wave_check, indent=2))

fig, ax = plt.subplots(figsize=(6,5))
sc = ax.scatter(ex, ey, c=psf_sums, s=18, cmap='viridis')
ax.set_title('Embedded ePSF zone centres'); ax.set_xlabel('XCENTER (parent detector pixels)'); ax.set_ylabel('YCENTER')
plt.colorbar(sc, ax=ax, label='ePSF sum'); plt.tight_layout(); plt.savefig(OUT/'epsf_zone_grid.png', dpi=160); plt.show()

## 4. Obtain cutouts with decoded FLAGS and multiple real observations

The full MEF in the uploaded bundle stores `FLAGS` as a compressed variable-length binary-table extension. IRSA's cutout service returns `FLAGS` as a 2-D image plane for the cutout. This cell follows IRSA's published TAP query pattern and downloads a small set of detector-matched DR1 cutouts from independent fields.

Edit `FIELDS` to add more regions. For a defensible real-sky validation, aim for **at least 3 widely separated fields, 20 or more distinct exposures, and more than one ePSF spatial zone**. The notebook will report the achieved counts instead of pretending these thresholds were met.

In [ ]:
# Field centers are editable. The first two come from the uploaded manifest / MEF metadata.
FIELDS = {
    'manifest_target': (304.693508808, 42.4436872991),
    'mef_pointing_center': (303.877716814435, 40.54668985260471),
    'm101_control': (210.80227, 54.34895),
    'm31_crowded_control': (10.6847, 41.2692),
}
DETECTOR = int(image_header.get('DETECTOR', 1))
CUTOUT_SIZE_DEG = 0.18
MAX_EXPOSURES_PER_FIELD = 5

from astropy.utils.data import conf
conf.remote_timeout = 240

def query_irsa_cutout_rows(ra, dec, size_deg=CUTOUT_SIZE_DEG, detector=DETECTOR):
    query = f"""
    SELECT
      'https://irsa.ipac.caltech.edu/' || a.uri || '?center={ra},{dec}d&size={size_deg}' AS uri,
      p.time_bounds_lower, p.energy_bandpassname, p.time_bounds_upper, p.provenance_version
    FROM spherex.artifact a
    JOIN spherex.plane p ON a.planeid = p.planeid
    WHERE 1 = CONTAINS(POINT('ICRS', {ra}, {dec}), p.poly)
    ORDER BY p.time_bounds_lower
    """
    tab = Irsa.query_tap(query).to_table()
    df = tab.to_pandas()
    if df.empty: return df
    df['uri'] = df['uri'].astype(str)
    df = df[df['uri'].str.contains('/dr1/', case=False, regex=False)]
    df = df[df['uri'].str.contains(f'D{detector}_spx_l2b', case=False, regex=False)]
    return df.sort_values('time_bounds_lower').drop_duplicates('uri').reset_index(drop=True)

def choose_evenly_spaced(df, n):
    if len(df) <= n: return df.copy()
    idx = np.unique(np.round(np.linspace(0, len(df)-1, n)).astype(int))
    return df.iloc[idx].copy()

def download_cutout(url, path, retries=3):
    last = None
    for k in range(retries):
        try:
            with requests.get(url, stream=True, timeout=240) as r:
                r.raise_for_status()
                tmp = Path(str(path)+'.part')
                with open(tmp,'wb') as f:
                    for chunk in r.iter_content(1024*1024):
                        if chunk: f.write(chunk)
                tmp.replace(path)
            return path
        except Exception as e:
            last=e; time.sleep(4*(k+1))
    raise last

cutout_records=[]
for field_name, (ra,dec) in FIELDS.items():
    print(f'Querying {field_name}: RA={ra}, Dec={dec}')
    try:
        df=query_irsa_cutout_rows(ra,dec)
        print('DR1 detector-matched rows:', len(df))
        chosen=choose_evenly_spaced(df, MAX_EXPOSURES_PER_FIELD)
        for j,row in chosen.iterrows():
            url=row['uri']
            fname=f'{field_name}_{len(cutout_records):03d}.fits'
            dest=RAW/fname
            if not dest.exists(): download_cutout(url,dest)
            cutout_records.append({'field':field_name,'path':str(dest),'url':url,
                                   'time_bounds_lower':str(row.get('time_bounds_lower')),
                                   'time_bounds_upper':str(row.get('time_bounds_upper')),
                                   'bandpass':str(row.get('energy_bandpassname')),
                                   'provenance_version':str(row.get('provenance_version'))})
    except Exception as e:
        print('FIELD QUERY/DOWNLOAD FAILED:', repr(e))

pd.DataFrame(cutout_records).to_csv(OUT/'cutout_manifest.csv',index=False)
print('\nTotal cutouts retrieved:',len(cutout_records))
print(pd.DataFrame(cutout_records).groupby('field').size() if cutout_records else 'No cutouts downloaded')

## 5. Discover detector-matched calibration candidates

Calibration files are not exposed through the SIA2 image search; IRSA documents direct directory browsing for these families. This cell lists candidate files for the MEF's detector. It **does not assume the newest candidate is the correct one**. Download candidates only after checking header/provenance against the MEF and the relevant pipeline release.

Required for wavelength science: **stand-alone Spectral WCS with `CWAVE` and `CBAND`**. Helpful/required for the requested full audit: Absolute Gain Matrix, Nonfunctional Pixel Map. The standalone ePSF calibration is optional for this 7.0.5 MEF because its ePSF grid is embedded, but it is useful for comparison. A Solid Angle Pixel Map is strongly recommended for accurate flux integration.

In [ ]:
IRSA_ROOT='https://irsa.ipac.caltech.edu/ibe/data/spherex/dr1/'
CAL_FAMILIES={
    'absolute_gain_matrix':'abs_gain_matrix/',
    'spectral_wcs':'spectral_wcs/',
    'epsf':'epsf/',
    'nonfunctional_mask':'nonfunc_v2/',
    'solid_angle_pixel_map':'solid_angle_pixel_map/',
}

def directory_links(url):
    r=requests.get(url,timeout=90); r.raise_for_status()
    soup=BeautifulSoup(r.text,'html.parser')
    out=[]
    for a in soup.find_all('a',href=True):
        href=a['href'].strip()
        if href in ('../','./','/'): continue
        full=urljoin(url,href)
        if href.endswith('/') and full.startswith(url): out.append(full)
    return sorted(set(out))

def fits_links(url):
    r=requests.get(url,timeout=90); r.raise_for_status()
    soup=BeautifulSoup(r.text,'html.parser')
    return sorted({urljoin(url,a['href'].strip()) for a in soup.find_all('a',href=True)
                   if a['href'].lower().split('?')[0].endswith(('.fits','.fit','.fits.gz'))})

calibration_candidates={}
for family,sub in CAL_FAMILIES.items():
    root=urljoin(IRSA_ROOT,sub)
    try:
        product_dirs=directory_links(root)
        found=[]
        for pdir in product_dirs:
            det_url=urljoin(pdir,f'{DETECTOR}/')
            try:
                for f in fits_links(det_url): found.append(f)
            except Exception:
                continue
        calibration_candidates[family]=sorted(set(found))
        print(f'\n{family}: {len(calibration_candidates[family])} FITS candidates')
        for i,url in enumerate(calibration_candidates[family][:30]): print(f'  [{i}] {url}')
    except Exception as e:
        calibration_candidates[family]=[]
        print(f'\n{family}: directory listing failed:',repr(e))

# Set explicit zero-based indices only after inspecting the listing above.
# Leaving a value as None means: do not auto-select an unverified calibration.
CALIBRATION_SELECTION = {
    'absolute_gain_matrix': None,
    'spectral_wcs': None,
    'epsf': None,
    'nonfunctional_mask': None,
    'solid_angle_pixel_map': None,
}

def download_selected_calibrations():
    downloaded=[]
    for family,idx in CALIBRATION_SELECTION.items():
        if idx is None: continue
        urls=calibration_candidates.get(family,[])
        if idx<0 or idx>=len(urls): raise IndexError((family,idx,len(urls)))
        url=urls[idx]
        name=Path(urlparse(url).path).name
        dest=CAL/(family+'__'+name)
        if not dest.exists(): download_cutout(url,dest)
        downloaded.append({'family':family,'url':url,'path':str(dest),'sha256':sha256_file(dest)})
    pd.DataFrame(downloaded).to_csv(OUT/'downloaded_calibrations.csv',index=False)
    return downloaded

chosen_calibrations=download_selected_calibrations()
print('Downloaded explicitly selected calibration files:',len(chosen_calibrations))

In [ ]:
# Optional fallback: attach calibration FITS as another Kaggle Dataset via Add Input.
# Kaggle inputs are read-only, so copy selected files into /kaggle/working.
manual_cal_files=[]
if KAGGLE_INPUT.exists():
    tokens=('abs_gain_matrix','spectral_wcs','cal-epsf','epsf_','nonfunc_v2','solid_angle_pixel_map')
    for src in KAGGLE_INPUT.rglob('*'):
        if src.is_file() and src.name.lower().endswith(('.fits','.fit','.fits.gz')) and any(t in src.name.lower() for t in tokens):
            dest=CAL/src.name
            if not dest.exists(): shutil.copy2(src,dest)
            manual_cal_files.append(dest)
    for src in KAGGLE_INPUT.rglob('*.zip'):
        if any(t in src.name.lower() for t in ('calibration','calibrations','spherex_cal')):
            destdir=CAL/(src.stem+'_manual'); destdir.mkdir(parents=True,exist_ok=True)
            try:
                with zipfile.ZipFile(src) as zf: zf.extractall(destdir)
            except Exception as exc: print('Could not extract calibration archive:',src,repr(exc))
n_cal=len(list(CAL.rglob('*.fits')))+len(list(CAL.rglob('*.fits.gz')))
print('Calibration FITS available locally:',n_cal)
if n_cal==0: print('No manual calibration FITS attached. Continue with IRSA discovery/download if Internet is enabled.')


### Calibration compatibility audit

Set `CALIBRATION_SELECTION` above and rerun the previous cell after viewing the candidate index. The following cell inspects actual downloaded FITS headers and data planes. A detector match is necessary but not sufficient: an exact provenance/version relationship must be demonstrated before declaring a calibration compatible.

In [ ]:
def inspect_calibration_file(path):
    row={'file':Path(path).name,'path':str(path),'sha256':sha256_file(path)}
    try:
        with fits.open(path,memmap=True,lazy_load_hdus=False) as hdul:
            row['hdus']=[{'name':h.name,'type':type(h).__name__,'shape':getattr(h.data,'shape',None)} for h in hdul]
            detector_values=[]
            for h in hdul:
                for k in ['DETECTOR','DETCOORD','VERSION','DATE','DATE-OBS','PROVVER','PROVENANCE','CALVER','BUNIT','DATEOBS','CAL_DATE']:
                    if k in h.header: row[f'{h.name}_{k}']=str(h.header[k])
                if 'DETECTOR' in h.header: detector_values.append(str(h.header['DETECTOR']))
                if isinstance(h,fits.BinTableHDU): row[f'{h.name}_columns']=list(h.columns.names)
                if getattr(h.data,'ndim',0)==2:
                    arr=np.asarray(h.data)
                    finite=np.isfinite(arr)
                    if finite.any():
                        vals=arr[finite]
                        row[f'{h.name}_array_audit']={'shape':list(arr.shape),'dtype':str(arr.dtype),
                            'finite_fraction':float(finite.mean()),'min':float(np.min(vals)),
                            'median':float(np.median(vals)),'p99':float(np.percentile(vals,99)),
                            'max':float(np.max(vals)),'nonzero_fraction':float(np.count_nonzero(vals)/vals.size)}
                        if 'nonfunc' in str(path).lower():
                            row[f'{h.name}_nonfunctional_pixels_value1']=int(np.count_nonzero(arr==1))
            row['detector_header_values']=sorted(set(detector_values))
            row['detector_header_match']=all(int(x)==DETECTOR for x in detector_values) if detector_values else None
    except Exception as e:
        row['open_error']=repr(e)
    return row

cal_rows=[]
for p in sorted(CAL.rglob('*.fits*')):
    cal_rows.append(inspect_calibration_file(p))
cal_df=pd.DataFrame(cal_rows)
cal_df.to_csv(OUT/'calibration_header_and_array_audit.csv',index=False)
if len(cal_df): display(cal_df)
else: print('No standalone calibration FITS files downloaded yet. Use directory listing, select explicit candidates, and rerun this cell.')

# Inspect science-grade Spectral WCS and compare it against the exposure's embedded ePSF-zone wavelength metadata.
from scipy.ndimage import map_coordinates
spectral_wcs_valid_files=[]
spectral_wcs_match_results=[]
for p in sorted(CAL.rglob('*.fits*')):
    if 'spectral_wcs' not in str(p).lower() and 'spectralwcs' not in str(p).lower() and 'wcs' not in p.name.lower():
        continue
    try:
        with fits.open(p,memmap=True) as h:
            print('\nSpectral WCS product:',p.name,'Extensions:',[x.name for x in h])
            if not ('CWAVE' in h and 'CBAND' in h):
                print('No CWAVE+CBAND pair; not a qualifying science-grade Spectral WCS file.'); continue
            cw=np.asarray(h['CWAVE'].data,float); cb=np.asarray(h['CBAND'].data,float)
            print('CWAVE/CBAND shapes:',cw.shape,cb.shape,'finite:',np.isfinite(cw).mean(),np.isfinite(cb).mean())
            if cw.ndim!=2 or cb.shape!=cw.shape or cw.shape!=(2040,2040):
                print('Wrong dimensions; not accepted as full detector calibration.'); continue
            spectral_wcs_valid_files.append(str(p))
            # ePSF zone centers are 0-based parent-detector pixel coordinates. Sample the 2-D calibration maps there.
            ex=np.asarray(epsf_tab['XCENTER'],float); ey=np.asarray(epsf_tab['YCENTER'],float)
            sample_cw=map_coordinates(cw,[ey,ex],order=1,mode='nearest')
            sample_cb=map_coordinates(cb,[ey,ex],order=1,mode='nearest')
            d_cw=sample_cw-np.asarray(epsf_tab['CWAVE'],float)
            d_cb=sample_cb-np.asarray(epsf_tab['CBAND'],float)
            result={'file':str(p),'sha256':sha256_file(p),'expected_detector':DETECTOR,
                    'detector_header':h['CWAVE'].header.get('DETECTOR',h[0].header.get('DETECTOR')),
                    'detcoord':h['CWAVE'].header.get('DETCOORD',h[0].header.get('DETCOORD')),
                    'n_zones_compared':int(len(ex)),
                    'cwave_delta_micron_median':float(np.nanmedian(d_cw)),
                    'cwave_delta_micron_p95_abs':float(np.nanpercentile(np.abs(d_cw),95)),
                    'cwave_delta_micron_max_abs':float(np.nanmax(np.abs(d_cw))),
                    'cband_delta_micron_median':float(np.nanmedian(d_cb)),
                    'cband_delta_micron_p95_abs':float(np.nanpercentile(np.abs(d_cb),95)),
                    'cband_delta_micron_max_abs':float(np.nanmax(np.abs(d_cb))),
                    'interpretation':'consistency comparison to embedded ePSF-zone wavelength metadata; not an independent spectroscopic validation'}
            spectral_wcs_match_results.append(result)
            print('ePSF-zone consistency:',result)
    except Exception as e:
        print('Spectral WCS audit error:',p.name,repr(e))
(OUT/'spectral_wcs_embedded_epsf_consistency.json').write_text(json.dumps(spectral_wcs_match_results,indent=2))


## 6. Gaia DR3 cross-check and empirical ePSF alignment

For every cutout, the notebook uses that exact exposure's embedded ePSF table. It maps Gaia star coordinates from cutout pixels to parent detector pixels with the IRSA-documented `CRPIX1A` / `CRPIX2A` keywords, then selects the nearest ePSF zone. This avoids using the one uploaded MEF's WCS/ePSF for unrelated exposures. Missing parent-pixel keywords cause a recorded failure rather than a guessed zone.

The output reports each star's pixel residual and approximate angular residual. A small median alone does not establish detector-wide alignment; inspect zone/field trends, reject poor fits, and test held-out fields.


In [ ]:
PHOT_REJECT_BITS=[2,6,9,10,11,15,17]  # documented photometry rejection bits


def bitmask_from_bits(flags_arr,bits):
    f=np.asarray(flags_arr,dtype=np.uint64)
    bad=np.zeros(f.shape,dtype=bool)
    for b in bits: bad |= ((f >> b) & 1).astype(bool)
    return bad


def parent_xy_from_cutout(x_cut,y_cut,image_header):
    """Convert 0-based cutout pixel coordinates to parent detector coordinates.

    This follows the IRSA tutorial: x_parent = 1 + x_cutout - CRPIX1A,
    y_parent = 1 + y_cutout - CRPIX2A. Those A-suffixed keywords are
    required; do not substitute the cutout WCS CRPIX1/2 values.
    """
    if 'CRPIX1A' not in image_header or 'CRPIX2A' not in image_header:
        raise KeyError('IRSA parent-coordinate keywords CRPIX1A/CRPIX2A are missing')
    return (1.0+np.asarray(x_cut,float)-float(image_header['CRPIX1A']),
            1.0+np.asarray(y_cut,float)-float(image_header['CRPIX2A']))


def get_epsf_row(table, x_parent, y_parent):
    d2=(np.asarray(table['XCENTER'],float)-x_parent)**2 + (np.asarray(table['YCENTER'],float)-y_parent)**2
    return int(np.argmin(d2))


def render_epsf_stamp(model,x0,y0,stamp_size=11):
    half=(stamp_size-1)/2
    x_ll=int(np.floor(float(x0)-half)); y_ll=int(np.floor(float(y0)-half))
    xs=x_ll+np.arange(stamp_size); ys=y_ll+np.arange(stamp_size)
    Y,X=np.meshgrid(ys,xs,indexing='ij')
    psf=np.asarray(model.evaluate(X,Y,flux=1.0,x_0=float(x0),y_0=float(y0)),float)
    s=np.nansum(psf)
    if not np.isfinite(s) or s<=0: raise ValueError('Rendered ePSF has non-positive normalization')
    return psf/s,x_ll,y_ll


def fit_single_epsf(data,var,flags,epsf_img,ovs,x_init,y_init,stamp_size=13):
    model=ImagePSF(data=np.asarray(epsf_img,float),oversampling=ovs)
    half=stamp_size//2
    xbase=int(round(x_init)); ybase=int(round(y_init))
    x1=xbase-half; y1=ybase-half
    if x1<0 or y1<0 or x1+stamp_size>data.shape[1] or y1+stamp_size>data.shape[0]: return None
    yy,xx=np.mgrid[y1:y1+stamp_size,x1:x1+stamp_size]
    d=np.asarray(data[y1:y1+stamp_size,x1:x1+stamp_size],float)
    vv=np.asarray(var[y1:y1+stamp_size,x1:x1+stamp_size],float)
    ff=np.asarray(flags[y1:y1+stamp_size,x1:x1+stamp_size],np.uint64)
    bad=bitmask_from_bits(ff,PHOT_REJECT_BITS) | ~np.isfinite(d) | ~np.isfinite(vv) | (vv<=0)
    good=~bad
    if good.sum()<50: return None
    border=np.zeros(d.shape,bool); border[[0,-1],:]=True; border[:,[0,-1]]=True
    bg0=float(np.nanmedian(d[border & good])) if np.any(border & good) else float(np.nanmedian(d[good]))
    def resid(p):
        amp,x0,y0,bg=p
        mod=model.evaluate(xx,yy,flux=amp,x_0=x0,y_0=y0)+bg
        return ((d-mod)/np.sqrt(vv))[good]
    p0=[max(float(np.nanmax(d[good])-bg0),1e-6),float(x_init),float(y_init),bg0]
    res=least_squares(resid,p0,bounds=([0,x_init-2,y_init-2,-np.inf],[np.inf,x_init+2,y_init+2,np.inf]),max_nfev=250)
    if not res.success: return None
    amp,xfit,yfit,bg=res.x
    return {'x_fit':float(xfit),'y_fit':float(yfit),'amplitude':float(amp),'background':float(bg),
            'chi2_red':float(np.mean(res.fun**2)),'n_valid':int(good.sum())}


alignment_rows=[]
gaia_cache={}  # one Gaia DR3 query per field; reuse across exposures
for rec in cutout_records:
    path=Path(rec['path'])
    try:
        with fits.open(path,memmap=False) as ch:
            if not all(n in ch for n in ['IMAGE','FLAGS','VARIANCE','EPSF']):
                print('Missing required cutout planes:',path.name,[h.name for h in ch]); continue
            cut=np.asarray(ch['IMAGE'].data,float)
            varc=np.asarray(ch['VARIANCE'].data,float)
            flagc=np.asarray(ch['FLAGS'].data)
            if flagc.ndim!=2 or flagc.shape!=cut.shape:
                print('FLAGS is not a decoded 2D image plane:',path.name,flagc.shape); continue
            zc=np.asarray(ch['ZODI'].data,float) if 'ZODI' in ch else np.zeros_like(cut)
            imh=ch['IMAGE'].header.copy()
            cutw=WCS(imh)
            epsft=Table(ch['EPSF'].data)  # ePSF belongs to this exact exposure/cutout
            epsfh=ch['EPSF'].header.copy()
            obsdate=imh.get('DATE-AVG')
        ra0,dec0=FIELDS[rec['field']]
        if rec['field'] not in gaia_cache:
            q=f"""SELECT TOP 3000 source_id,ra,dec,pmra,pmdec,ra_error,dec_error,phot_g_mean_mag,ruwe,astrometric_params_solved
            FROM gaiadr3.gaia_source
            WHERE CONTAINS(POINT('ICRS',ra,dec),CIRCLE('ICRS',{ra0},{dec0},0.14))=1
            AND phot_g_mean_mag < 17.5 AND (ruwe IS NULL OR ruwe < 1.4)
            AND astrometric_params_solved IN (31,95) AND pmra IS NOT NULL AND pmdec IS NOT NULL"""
            job=Gaia.launch_job_async(q,dump_to_file=False)
            gaia_cache[rec['field']]=job.get_results()
        gt=gaia_cache[rec['field']]
        if len(gt)==0: print('No Gaia stars in',path.name); continue
        stars=SkyCoord(ra=np.asarray(gt['ra'])*u.deg,dec=np.asarray(gt['dec'])*u.deg,
                       pm_ra_cosdec=np.nan_to_num(np.asarray(gt['pmra'],float))*u.mas/u.yr,
                       pm_dec=np.nan_to_num(np.asarray(gt['pmdec'],float))*u.mas/u.yr,
                       obstime=Time(2016.0,format='jyear'),frame='icrs')
        obst=Time(obsdate) if obsdate else Time('2025-04-29T00:25:25')
        stars_epoch=stars.apply_space_motion(new_obstime=obst)
        xloc,yloc=cutw.world_to_pixel(stars_epoch)
        xpar,ypar=parent_xy_from_cutout(xloc,yloc,imh)
        gmag=np.asarray(gt['phot_g_mean_mag'],float)
        ov=(int(epsfh.get('OVSMPY',5)),int(epsfh.get('OVSMPX',5)))
        for k in range(len(gt)):
            xl,yl=float(xloc[k]),float(yloc[k])
            if not (8<xl<cut.shape[1]-8 and 8<yl<cut.shape[0]-8): continue
            if not np.isfinite(xpar[k]+ypar[k]): continue
            idx=get_epsf_row(epsft,float(xpar[k]),float(ypar[k]))
            eimg=np.asarray(epsft['EPSF'][idx],float).reshape(33,33)
            fit=fit_single_epsf(cut-zc,varc,flagc,eimg,ov,xl,yl)
            if fit is None: continue
            dx=fit['x_fit']-xl; dy=fit['y_fit']-yl
            alignment_rows.append({'field':rec['field'],'cutout':path.name,'source_id':str(gt['source_id'][k]),
                                   'gmag':float(gmag[k]),'x_gaia':xl,'y_gaia':yl,
                                   'x_parent':float(xpar[k]),'y_parent':float(ypar[k]),
                                   'x_fit':fit['x_fit'],'y_fit':fit['y_fit'],'dx_px':dx,'dy_px':dy,
                                   'radial_px':float(np.hypot(dx,dy)),'fit_chi2_red':fit['chi2_red'],
                                   'epsf_zone':idx,'epsf_bin_x':int(epsft['BINX'][idx]),'epsf_bin_y':int(epsft['BINY'][idx]),
                                   'epsf_cwave_um':float(epsft['CWAVE'][idx]),
                                   'parent_coord_method':'IRSA_CRPIX1A_CRPIX2A'})
    except Exception as e:
        print('Alignment cutout failed:',path.name,repr(e))

align_df=pd.DataFrame(alignment_rows)
align_df.to_csv(OUT/'gaia_epsf_alignment_residuals.csv',index=False)
if len(align_df):
    from astropy.wcs.utils import proj_plane_pixel_scales
    scales=[]
    for rec in cutout_records:
        try:
            with fits.open(rec['path'],memmap=True) as h:
                scales.append(float(np.mean(proj_plane_pixel_scales(WCS(h['IMAGE'].header).celestial))*3600))
        except Exception: pass
    scale_arcsec=float(np.nanmedian(scales)) if scales else float('nan')
    summary={'n_star_fits':int(len(align_df)),'n_fields':int(align_df.field.nunique()),
             'n_cutouts':int(align_df.cutout.nunique()),'pixel_scale_arcsec_approx':scale_arcsec,
             'dx_median_px':float(align_df.dx_px.median()),'dy_median_px':float(align_df.dy_px.median()),
             'radial_median_px':float(align_df.radial_px.median()),'radial_p95_px':float(align_df.radial_px.quantile(.95)),
             'radial_median_arcsec_approx':float(align_df.radial_px.median()*scale_arcsec),
             'radial_p95_arcsec_approx':float(align_df.radial_px.quantile(.95)*scale_arcsec),
             'qualification':'per-exposure ePSF zone selected using IRSA parent-pixel mapping; successful Gaia-star fits only; inspect field/zone residual trends and held-out fields before global claims'}
    (OUT/'gaia_epsf_alignment_summary.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2))
else:
    print('No successful Gaia+ePSF fits. Alignment is NOT validated; inspect query results, flag decoding, parent WCS and ePSF zones.')


## 7. Real-sky injection/recovery benchmark

The workflow injects artificial point sources into **real SPHEREx residual images** (`IMAGE - ZODI`) using the ePSF embedded in each exact cutout. It selects the spatial ePSF zone separately for every injection using the official parent-coordinate mapping. It uses per-pixel `VARIANCE`, documented photometry-rejection flags, and the detector-specific Solid Angle Pixel Map (SAPM) when selected and shape-compatible. Otherwise it labels the use of `OMEGA_MEDIAN` as a fallback.

This is a local ePSF matched-filter recovery experiment, not yet an end-to-end blind catalogue benchmark. Blank-position controls report threshold exceedance at selected blank positions, not a definitive false-positive rate. Survey-wide completeness requires multi-field held-out testing, sufficient independent exposures, and a frozen blind detection pipeline.


In [ ]:
from scipy.ndimage import maximum_filter

FLUX_BINS_MJY = np.array([0.05,0.1,0.2,0.4,0.8,1.6,3.2,6.4])
N_INJECT_PER_BIN_PER_CUTOUT = 35
N_FALSE_CONTROLS_PER_CUTOUT = 60
DETECTION_SNR_THRESHOLD = 5.0
STAMP_SIZE = 11
PHOT_REJECT_BITS=[2,6,9,10,11,15,17]
SELECTION_EXTRA_BITS=[0,21]  # TRANSIENT and SOURCE: avoid these positions when injecting


def wilson_interval(k,n,z=1.959963984540054):
    if n==0: return (np.nan,np.nan)
    p=k/n; den=1+z*z/n
    center=(p+z*z/(2*n))/den
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return center-half,center+half


def local_score(data,var,flags,model,x0,y0,area_sr=None,stamp_size=STAMP_SIZE):
    psf,xll,yll=render_epsf_stamp(model,x0,y0,stamp_size)
    xur=xll+stamp_size; yur=yll+stamp_size
    if xll<0 or yll<0 or xur>data.shape[1] or yur>data.shape[0]: return None
    d=np.asarray(data[yll:yur,xll:xur],float)
    v=np.asarray(var[yll:yur,xll:xur],float)
    f=np.asarray(flags[yll:yur,xll:xur],np.uint64)
    om=np.asarray(area_sr[yll:yur,xll:xur],float) if area_sr is not None else np.ones_like(d)
    bad=bitmask_from_bits(f,PHOT_REJECT_BITS) | ~np.isfinite(d) | ~np.isfinite(v) | (v<=0) | ~np.isfinite(om) | (om<=0)
    good=~bad
    if good.sum()<0.8*stamp_size*stamp_size: return None
    bg=float(np.nanmedian(d[good]))
    # Surface-brightness template per unit integrated flux (MJy).
    p=psf[good]/om[good]; w=1/v[good]; r=d[good]-bg
    den=float(np.sum(p*p*w))
    if den<=0: return None
    amp=float(np.sum(p*r*w)/den)
    snr=float(np.sum(p*r*w)/np.sqrt(den))
    return {'snr':snr,'amp_mjy':amp,'psf':psf,'xll':xll,'yll':yll}


def apply_stamp(arr,stamp,xll,yll,scale):
    arr[yll:yll+stamp.shape[0],xll:xll+stamp.shape[1]] += stamp*scale


def get_pixel_area_sr(header, fallback_header=None):
    omega=header.get('OMEGA_MEDIAN')
    source='CUTOUT_OMEGA_MEDIAN'
    if omega is None and fallback_header is not None:
        omega=fallback_header.get('OMEGA_MEDIAN'); source='PARENT_MEF_OMEGA_MEDIAN_FALLBACK'
    if omega is None or not np.isfinite(float(omega)) or float(omega)<=0:
        return None, 'NO_VALID_OMEGA_MEDIAN'
    return float(omega)/(206264.806247**2), source


# Prefer the detector-specific SAPM when an explicitly selected and shape-compatible file exists.
sapm_data=None; sapm_file=None; sapm_header=None
for p in sorted(CAL.rglob('*.fits*')):
    if 'solid_angle_pixel_map' not in str(p).lower() and 'sapm' not in p.name.lower():
        continue
    try:
        with fits.open(p,memmap=True) as hh:
            det_ok=True
            hdet=hh[0].header.get('DETECTOR')
            if hdet is not None: det_ok=int(hdet)==DETECTOR
            candidates=[h for h in hh if getattr(h.data,'ndim',0)==2 and getattr(h.data,'shape',None)==(2040,2040)]
            if det_ok and candidates:
                sapm_data=np.asarray(candidates[0].data,float).copy()
                sapm_header=candidates[0].header.copy(); sapm_file=str(p)
                break
    except Exception as e: print('SAPM inspect failed:',p.name,repr(e))
if sapm_data is not None:
    print('Using detector-matched 2040x2040 SAPM:',sapm_file)
else:
    print('No selected detector-matched SAPM found; per-cutout OMEGA_MEDIAN fallback will be used and labelled.')


def area_map_for_cutout(shape,image_header,epsf_header):
    if sapm_data is not None:
        yy,xx=np.indices(shape,dtype=float)
        xp,yp=parent_xy_from_cutout(xx,yy,image_header)
        xi=np.rint(xp).astype(int); yi=np.rint(yp).astype(int)
        inside=(xi>=0)&(xi<2040)&(yi>=0)&(yi<2040)
        amap=np.full(shape,np.nan,float)
        amap[inside]=sapm_data[yi[inside],xi[inside]]/(206264.806247**2)
        if np.isfinite(amap).mean()>0.98 and np.nanmedian(amap)>0:
            return amap,'DETECTOR_SAPM'
        print('SAPM did not map cleanly to this cutout; using OMEGA_MEDIAN fallback.')
    omega_sr,source=get_pixel_area_sr(image_header,globals().get('image_header'))
    if omega_sr is None:
        return None,source
    return np.full(shape,omega_sr,float),source


recovery_rows=[]
for rec in tqdm(cutout_records,desc='Real-sky injection/recovery'):
    path=Path(rec['path'])
    try:
        with fits.open(path,memmap=False) as ch:
            if not all(n in ch for n in ['IMAGE','VARIANCE','FLAGS','EPSF']): continue
            dat=np.asarray(ch['IMAGE'].data,float)
            varc=np.asarray(ch['VARIANCE'].data,float)
            flags=np.asarray(ch['FLAGS'].data)
            if flags.ndim!=2 or flags.shape!=dat.shape: continue
            z=np.asarray(ch['ZODI'].data,float) if 'ZODI' in ch else np.zeros_like(dat)
            header=ch['IMAGE'].header.copy()
            epsft=Table(ch['EPSF'].data)
            epsfh=ch['EPSF'].header.copy()
        if dat.size<1000: continue
        area_sr,area_source=area_map_for_cutout(dat.shape,header,epsfh)
        if area_sr is None:
            print('No valid pixel solid angle in',path.name,'; skip absolute mJy injection.')
            continue
        clean=~bitmask_from_bits(flags,PHOT_REJECT_BITS+SELECTION_EXTRA_BITS)
        clean &= np.isfinite(dat)&np.isfinite(varc)&(varc>0)&np.isfinite(area_sr)&(area_sr>0)
        residual=dat-z
        from scipy.ndimage import binary_erosion
        valid_centers=binary_erosion(clean,structure=np.ones((STAMP_SIZE,STAMP_SIZE)),border_value=0)
        yyv,xxv=np.where(valid_centers)
        if len(xxv)<100: print('Too few clean injection sites in',path.name); continue
        rng=np.random.default_rng(int(hashlib.sha256(path.name.encode()).hexdigest()[:8],16))
        ov=(int(epsfh.get('OVSMPY',5)),int(epsfh.get('OVSMPX',5)))
        crpix_header=header
        # Blank-position controls before adding anything; this estimates threshold exceedance in selected blank sites.
        ctrl_order=rng.permutation(len(xxv)); controls=[]
        for ii in ctrl_order:
            x0=float(xxv[ii]); y0=float(yyv[ii])
            if all(np.hypot(x0-x1,y0-y1)>=STAMP_SIZE+2 for x1,y1 in controls): controls.append((x0,y0))
            if len(controls)>=N_FALSE_CONTROLS_PER_CUTOUT: break
        for x0,y0 in controls:
            xp,yp=parent_xy_from_cutout(x0,y0,crpix_header)
            zidx=get_epsf_row(epsft,float(xp),float(yp))
            eimg=np.asarray(epsft['EPSF'][zidx],float).reshape(33,33)
            model=ImagePSF(data=eimg,oversampling=ov)
            candidates=[]
            for dy in [-1,0,1]:
                for dx in [-1,0,1]:
                    s=local_score(residual,varc,flags,model,x0+dx,y0+dy,area_sr=area_sr)
                    if s is not None: candidates.append(s['snr'])
            if candidates:
                recovery_rows.append({'field':rec['field'],'cutout':path.name,'flux_injected_mjy':0.0,
                                      'recovered':bool(max(candidates)>=DETECTION_SNR_THRESHOLD),'snr':float(max(candidates)),
                                      'dx_best_px':np.nan,'dy_best_px':np.nan,'flux_recovered_mjy':np.nan,
                                      'fractional_flux_bias':np.nan,'epsf_zone':zidx,'pixel_area_source':area_source,
                                      'trial_type':'blank_position_threshold_control'})
        # Non-overlapping source injections. The local ePSF zone is selected separately for every input position.
        for flux_mjy in FLUX_BINS_MJY:
            chosen=[]; order=rng.permutation(len(xxv))
            for ii in order:
                x0=float(xxv[ii]); y0=float(yyv[ii])
                if all(np.hypot(x0-x1,y0-y1)>=STAMP_SIZE+2 for x1,y1 in chosen): chosen.append((x0,y0))
                if len(chosen)>=N_INJECT_PER_BIN_PER_CUTOUT: break
            for x0,y0 in chosen:
                xp,yp=parent_xy_from_cutout(x0,y0,crpix_header)
                zidx=get_epsf_row(epsft,float(xp),float(yp))
                eimg=np.asarray(epsft['EPSF'][zidx],float).reshape(33,33)
                model=ImagePSF(data=eimg,oversampling=ov)
                psf,xll,yll=render_epsf_stamp(model,x0,y0,STAMP_SIZE)
                test=residual.copy()
                omega_stamp=area_sr[yll:yll+STAMP_SIZE,xll:xll+STAMP_SIZE]
                if omega_stamp.shape!=psf.shape or not np.all(np.isfinite(omega_stamp)&(omega_stamp>0)): continue
                # Inject integrated source flux in MJy across pixels with spatially varying solid angle.
                apply_stamp(test,psf/omega_stamp,xll,yll,float(flux_mjy)*1e-9)
                candidates=[]
                for dy in [-1,0,1]:
                    for dx in [-1,0,1]:
                        s=local_score(test,varc,flags,model,x0+dx,y0+dy,area_sr=area_sr)
                        if s is not None: candidates.append((s['snr'],dx,dy,s['amp_mjy']))
                if not candidates: continue
                best=max(candidates,key=lambda q:q[0])
                amp_mjy=float(best[3]*1e9)
                recovery_rows.append({'field':rec['field'],'cutout':path.name,'flux_injected_mjy':float(flux_mjy),
                                      'recovered':bool(best[0]>=DETECTION_SNR_THRESHOLD),'snr':float(best[0]),
                                      'dx_best_px':int(best[1]),'dy_best_px':int(best[2]),
                                      'flux_recovered_mjy':amp_mjy,'fractional_flux_bias':float((amp_mjy-flux_mjy)/flux_mjy),
                                      'epsf_zone':zidx,'pixel_area_source':area_source,'trial_type':'local_matched_filter_recovery'})
    except Exception as e:
        print('Injection/recovery failure:',path.name,repr(e))

rec_df=pd.DataFrame(recovery_rows)
rec_df.to_csv(OUT/'injection_recovery_trials.csv',index=False)
if len(rec_df):
    inj=rec_df[(rec_df.flux_injected_mjy>0)&(rec_df.trial_type=='local_matched_filter_recovery')].copy()
    rows=[]
    for flux,g in inj.groupby('flux_injected_mjy'):
        n=len(g); k=int(g.recovered.sum()); lo,hi=wilson_interval(k,n)
        rows.append({'flux_injected_mjy':float(flux),'n_injected':n,'n_recovered':k,'completeness':k/n,
                     'wilson95_low':lo,'wilson95_high':hi,'median_snr':float(g.snr.median()),
                     'median_fractional_flux_bias':float(g.fractional_flux_bias.median()),
                     'p95_abs_fractional_flux_bias':float(g.fractional_flux_bias.abs().quantile(.95))})
    comp=pd.DataFrame(rows).sort_values('flux_injected_mjy')
    comp.to_csv(OUT/'completeness_curve.csv',index=False)
    ctrl=rec_df[rec_df.trial_type=='blank_position_threshold_control']
    summary={'n_cutouts_manifest':len(cutout_records),'n_cutouts_with_trials':int(rec_df.cutout.nunique()),
             'n_fields_with_trials':int(rec_df.field.nunique()),'n_injections':int(len(inj)),
             'n_blank_position_controls':int(len(ctrl)),
             'blank_position_threshold_exceedance_fraction':float(ctrl.recovered.mean()) if len(ctrl) else None,
             'threshold_snr':DETECTION_SNR_THRESHOLD,'flux_bins_mJy':FLUX_BINS_MJY.tolist(),
             'pixel_area_sources':sorted(inj.pixel_area_source.dropna().unique().tolist()),
             'qualification':'real-image local ePSF matched-filter injection/recovery; parent-coordinate zone selected per injection; not a blind catalogue completeness test; use SAPM for spatial flux scale where available'}
    (OUT/'injection_recovery_summary.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2)); display(comp)
    fig,ax=plt.subplots(figsize=(7,5))
    ax.errorbar(comp.flux_injected_mjy,comp.completeness,
                yerr=[comp.completeness-comp.wilson95_low,comp.wilson95_high-comp.completeness],fmt='o-',capsize=4)
    ax.set_xscale('log'); ax.set_ylim(-.03,1.03); ax.set_xlabel('Injected source flux (mJy)'); ax.set_ylabel('Recovery fraction')
    ax.set_title('SPHEREx DR1 real-image local matched-filter recovery'); ax.grid(alpha=.25)
    plt.tight_layout(); plt.savefig(OUT/'completeness_curve.png',dpi=170); plt.show()
else:
    print('No injection trials were produced. Ensure decoded 2-D FLAGS cutouts, CRPIX1A/2A and pixel-area metadata are available.')


## 7B. Blind full-cutout matched-filter recovery (stronger completeness test)

This second benchmark injects multiple artificial sources into each real cutout at once, runs a fixed PSF-matched filter over the **entire cutout without telling the detector where the injections are**, extracts local-maximum detections, and performs one-to-one positional matching to the hidden injection coordinates. Positions are restricted to the dominant ePSF zone in each cutout so one spatially appropriate ePSF can be used for the convolution kernel. This is still a defined matched-filter detection benchmark, not the mission pipeline or an all-sky completeness claim.


In [ ]:
from scipy.ndimage import binary_erosion
from scipy.signal import fftconvolve
from scipy.spatial import cKDTree
from scipy.optimize import linear_sum_assignment

BLIND_MATCH_RADIUS_PX = 2.0
BLIND_MIN_COVERAGE = 0.85


def run_blind_psf_filter(image,var,flags,kernel,threshold=DETECTION_SNR_THRESHOLD):
    good=(~bitmask_from_bits(flags,PHOT_REJECT_BITS) & np.isfinite(image) &
          np.isfinite(var) & (var>0))
    if good.sum()<kernel.size*5: return []
    bg=float(np.nanmedian(image[good]))
    weight=np.zeros_like(var,float)
    weight[good]=1.0/var[good]
    resid=np.where(good,image-bg,0.0)
    # Kernel is a surface-brightness model per unit integrated flux.
    kr=kernel[::-1,::-1]
    numerator=fftconvolve(resid*weight,kr,mode='same')
    denom2=fftconvolve(weight,kr**2,mode='same')
    snr=np.divide(numerator,np.sqrt(denom2),out=np.full_like(numerator,np.nan),where=denom2>0)
    coverage=fftconvolve(good.astype(float),np.ones(kernel.shape,float),mode='same')/kernel.size
    snr[coverage<BLIND_MIN_COVERAGE]=np.nan
    finite=np.isfinite(snr)
    localmax=maximum_filter(np.where(finite,snr,-np.inf),size=3,mode='constant',cval=-np.inf)
    yy,xx=np.where(finite & (snr>=threshold) & (snr==localmax))
    return [{'x':float(x),'y':float(y),'snr':float(snr[y,x])} for x,y in zip(xx,yy)]


blind_rows=[]
blind_control_rows=[]
for rec in tqdm(cutout_records,desc='Blind matched-filter benchmark'):
    path=Path(rec['path'])
    try:
        with fits.open(path,memmap=False) as ch:
            if not all(n in ch for n in ['IMAGE','VARIANCE','FLAGS','EPSF']): continue
            dat=np.asarray(ch['IMAGE'].data,float)
            varc=np.asarray(ch['VARIANCE'].data,float)
            flags=np.asarray(ch['FLAGS'].data)
            if flags.ndim!=2 or flags.shape!=dat.shape: continue
            z=np.asarray(ch['ZODI'].data,float) if 'ZODI' in ch else np.zeros_like(dat)
            header=ch['IMAGE'].header.copy()
            epsft=Table(ch['EPSF'].data)
            epsfh=ch['EPSF'].header.copy()
        area_sr,area_source=area_map_for_cutout(dat.shape,header,epsfh)
        if area_sr is None: continue
        clean=(~bitmask_from_bits(flags,PHOT_REJECT_BITS+SELECTION_EXTRA_BITS) &
               np.isfinite(dat)&np.isfinite(varc)&(varc>0)&np.isfinite(area_sr)&(area_sr>0))
        valid_centers=binary_erosion(clean,structure=np.ones((STAMP_SIZE,STAMP_SIZE)),border_value=0)
        yyv,xxv=np.where(valid_centers)
        if len(xxv)<100: continue
        # Map every valid cutout pixel to its parent-detector coordinate and nearest ePSF zone.
        yg,xg=np.indices(dat.shape,dtype=float)
        xpar,ypar=parent_xy_from_cutout(xg,yg,header)
        zone_centers=np.column_stack([np.asarray(epsft['XCENTER'],float),np.asarray(epsft['YCENTER'],float)])
        _,zone_map=cKDTree(zone_centers).query(np.column_stack([xpar.ravel(),ypar.ravel()]),k=1)
        zone_map=zone_map.reshape(dat.shape)
        zone_counts=np.bincount(zone_map[valid_centers].astype(int),minlength=len(epsft))
        zone_idx=int(np.argmax(zone_counts))
        candidates_y,candidates_x=np.where(valid_centers & (zone_map==zone_idx))
        if len(candidates_x)<N_INJECT_PER_BIN_PER_CUTOUT:
            print('Insufficient clean area in dominant ePSF zone:',path.name,'zone',zone_idx,'n',len(candidates_x)); continue
        rng=np.random.default_rng(int(hashlib.sha256((path.name+'blind').encode()).hexdigest()[:8],16))
        order=rng.permutation(len(candidates_x)); chosen=[]
        for ii in order:
            x0=float(candidates_x[ii]); y0=float(candidates_y[ii])
            if all(np.hypot(x0-x1,y0-y1)>=STAMP_SIZE+2 for x1,y1 in chosen): chosen.append((x0,y0))
            if len(chosen)>=N_INJECT_PER_BIN_PER_CUTOUT: break
        if len(chosen)<max(10,N_INJECT_PER_BIN_PER_CUTOUT//2):
            print('Not enough separated source locations:',path.name,len(chosen)); continue
        eimg=np.asarray(epsft['EPSF'][zone_idx],float).reshape(33,33)
        ovs=(int(epsfh.get('OVSMPY',5)),int(epsfh.get('OVSMPX',5)))
        model=ImagePSF(data=eimg,oversampling=ovs)
        # Central subpixel phase matches integer-pixel injection centers.
        psf_kernel,_,_=render_epsf_stamp(model,STAMP_SIZE//2,STAMP_SIZE//2,STAMP_SIZE)
        area_kernel=float(np.nanmedian(area_sr[valid_centers & (zone_map==zone_idx)]))
        if not np.isfinite(area_kernel) or area_kernel<=0: continue
        kernel=psf_kernel/area_kernel
        residual=dat-z
        baseline_detections=run_blind_psf_filter(residual,varc,flags,kernel)
        blind_control_rows.append({'field':rec['field'],'cutout':path.name,'epsf_zone':zone_idx,
                                   'n_blank_filter_detections':len(baseline_detections),
                                   'cutout_pixels':int(dat.size),'blank_detections_per_mpix':float(len(baseline_detections)/(dat.size/1e6)),
                                   'pixel_area_source':area_source})
        for flux_mjy in FLUX_BINS_MJY:
            test=residual.copy()
            injected=[]
            for x0,y0 in chosen:
                psf,xll,yll=render_epsf_stamp(model,x0,y0,STAMP_SIZE)
                omega=area_sr[yll:yll+STAMP_SIZE,xll:xll+STAMP_SIZE]
                if omega.shape!=psf.shape or not np.all(np.isfinite(omega)&(omega>0)): continue
                apply_stamp(test,psf/omega,xll,yll,float(flux_mjy)*1e-9)
                injected.append((x0,y0))
            detections=run_blind_psf_filter(test,varc,flags,kernel)
            pairs=[]
            if injected and detections:
                injxy=np.asarray(injected,float)
                detxy=np.asarray([[d['x'],d['y']] for d in detections],float)
                dist=np.sqrt(((injxy[:,None,:]-detxy[None,:,:])**2).sum(axis=2))
                rr,cc=linear_sum_assignment(dist)
                pairs=[(int(i),int(j),float(dist[i,j])) for i,j in zip(rr,cc) if dist[i,j]<=BLIND_MATCH_RADIUS_PX]
            matched={i:(j,d) for i,j,d in pairs}
            for i,(x0,y0) in enumerate(injected):
                if i in matched:
                    j,dsep=matched[i]; det=detections[j]
                    fit=local_score(test,varc,flags,model,det['x'],det['y'],area_sr=area_sr)
                    rec_mjy=float(fit['amp_mjy']*1e9) if fit else np.nan
                    blind_rows.append({'field':rec['field'],'cutout':path.name,'flux_injected_mjy':float(flux_mjy),
                                       'recovered':True,'n_detections_full_cutout':len(detections),'n_injected':len(injected),
                                       'match_sep_px':dsep,'detection_snr':det['snr'],'flux_recovered_mjy':rec_mjy,
                                       'fractional_flux_bias':float((rec_mjy-flux_mjy)/flux_mjy) if np.isfinite(rec_mjy) else np.nan,
                                       'epsf_zone':zone_idx,'pixel_area_source':area_source})
                else:
                    blind_rows.append({'field':rec['field'],'cutout':path.name,'flux_injected_mjy':float(flux_mjy),
                                       'recovered':False,'n_detections_full_cutout':len(detections),'n_injected':len(injected),
                                       'match_sep_px':np.nan,'detection_snr':np.nan,'flux_recovered_mjy':np.nan,
                                       'fractional_flux_bias':np.nan,'epsf_zone':zone_idx,'pixel_area_source':area_source})
    except Exception as e:
        print('Blind benchmark failed:',path.name,repr(e))

blind_df=pd.DataFrame(blind_rows)
blind_df.to_csv(OUT/'blind_matched_filter_trials.csv',index=False)
blind_ctrl_df=pd.DataFrame(blind_control_rows)
blind_ctrl_df.to_csv(OUT/'blind_matched_filter_blank_controls.csv',index=False)
if len(blind_df):
    rows=[]
    for flux,g in blind_df.groupby('flux_injected_mjy'):
        n=len(g); k=int(g.recovered.sum()); lo,hi=wilson_interval(k,n)
        rows.append({'flux_injected_mjy':float(flux),'n_injected':n,'n_recovered':k,'completeness':k/n,
                     'wilson95_low':lo,'wilson95_high':hi,'median_detection_snr':float(g.loc[g.recovered,'detection_snr'].median()) if k else np.nan,
                     'median_fractional_flux_bias':float(g.loc[g.recovered,'fractional_flux_bias'].median()) if k else np.nan})
    blind_comp=pd.DataFrame(rows).sort_values('flux_injected_mjy')
    blind_comp.to_csv(OUT/'blind_completeness_curve.csv',index=False)
    blind_summary={'n_cutouts':int(blind_df.cutout.nunique()),'n_fields':int(blind_df.field.nunique()),
                   'n_injections':int(len(blind_df)),'n_recovered':int(blind_df.recovered.sum()),
                   'median_blank_detections_per_mpix':float(blind_ctrl_df.blank_detections_per_mpix.median()) if len(blind_ctrl_df) else None,
                   'detection_threshold_snr':DETECTION_SNR_THRESHOLD,'one_to_one_match_radius_px':BLIND_MATCH_RADIUS_PX,
                   'qualification':'blind full-cutout PSF-matched filter benchmark with one zone-specific ePSF kernel per cutout; not the SPHEREx mission pipeline; validate on held-out fields and larger samples'}
    (OUT/'blind_completeness_summary.json').write_text(json.dumps(blind_summary,indent=2))
    display(blind_comp)
    fig,ax=plt.subplots(figsize=(7,5))
    ax.errorbar(blind_comp.flux_injected_mjy,blind_comp.completeness,
                yerr=[blind_comp.completeness-blind_comp.wilson95_low,blind_comp.wilson95_high-blind_comp.completeness],fmt='o-',capsize=4)
    ax.set_xscale('log'); ax.set_ylim(-.03,1.03); ax.set_xlabel('Injected source flux (mJy)'); ax.set_ylabel('Blind recovery fraction')
    ax.set_title('SPHEREx DR1 blind PSF-matched recovery'); ax.grid(alpha=.25)
    plt.tight_layout(); plt.savefig(OUT/'blind_completeness_curve.png',dpi=170); plt.show()
else:
    print('No blind trials produced. Check cutout geometry, CRPIX1A/2A, decoded FLAGS, SAPM/OMEGA and available clean zone area.')


## 8. Final reproducibility package

The output bundle includes the exact MEF byte hash, audit JSON, calibration candidate list, any downloaded calibration headers/hashes, cutout manifest, Gaia/ePSF residuals, injection trials, completeness curve and plots. It excludes the full 60 MB MEF and bulk raw cutouts by default; include them only when needed for independent re-analysis.

In [ ]:
# Preserve the calibration directory listing even if none were selected.
cal_listing=[]
for fam,urls in calibration_candidates.items():
    for i,url in enumerate(urls): cal_listing.append({'family':fam,'index':i,'url':url})
pd.DataFrame(cal_listing).to_csv(OUT/'calibration_candidates.csv',index=False)

run_meta={
 'input_bundle':ZIP_PATH.name if ZIP_PATH is not None else 'attached extracted Kaggle Dataset',
 'mef_file':MEF_PATH.name,
 'mef_sha256':MEF_SHA256,
 'mef_version':primary_header.get('VERSION'),
 'detector':DETECTOR,
 'obsid':image_header.get('OBSID'),
 'cutouts_requested':len(FIELDS)*MAX_EXPOSURES_PER_FIELD,
 'cutouts_downloaded':len(cutout_records),
 'alignment_fits':len(align_df),
 'injection_trials':len(rec_df),
 'blind_detection_trials':len(blind_df),
 'blind_detection_fields':int(blind_df.field.nunique()) if len(blind_df) else 0,
 'calibration_selection':CALIBRATION_SELECTION,
 'created_utc':Time.now().utc.isot,
 'runtime_claims':{
  'raw_mef_bytes_inspected':True,
  'standalone_spectral_wcs_validated':bool(globals().get('spectral_wcs_valid_files', [])),
  'spectral_wcs_epsf_zone_consistency_rows':len(globals().get('spectral_wcs_match_results', [])),
  'gaia_alignment_measured':len(align_df)>0,
  'real_image_injection_recovery_measured':len(rec_df)>0,
  'blind_full_cutout_recovery_measured':len(blind_df)>0,
  'survey_wide_completeness_validated':False,
  'citizen_science_study_conducted':False,
 }
}
(OUT/'run_metadata.json').write_text(json.dumps(run_meta,indent=2))

# Include selected calibration FITS in output bundle for provenance.
RESULT_ZIP=WORK/'spherex_real_validation_results.zip'
if RESULT_ZIP.exists(): RESULT_ZIP.unlink()
with zipfile.ZipFile(RESULT_ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob('*'):
        if p.is_file(): zf.write(p,p.relative_to(WORK))
    for p in CAL.rglob('*'):
        if p.is_file(): zf.write(p,p.relative_to(WORK))
print('Created:',RESULT_ZIP,'bytes:',RESULT_ZIP.stat().st_size)
print('Individual CSV/JSON/PNG results:',OUT)
print('Kaggle: save a notebook version; download the ZIP or individual files from its Output/Files panel.')
try:
    from IPython.display import FileLink, display
    display(FileLink(str(RESULT_ZIP)))
except Exception:
    pass

## 9. Interpretation checklist before any scientific claim

- [ ] At least 3 distinct sky fields were actually downloaded.
- [ ] At least 20 independent exposures were analysed (or the report explicitly says fewer).
- [ ] `FLAGS` is a decoded 2-D image for every benchmark cutout.
- [ ] Standalone Spectral WCS has both `CWAVE` and `CBAND`, correct detector, matching spatial orientation, and a provenance relationship established for the MEF processing lineage.
- [ ] Absolute Gain Matrix, nonfunctional map and optional SAPM have detector/provenance compatibility documented; none were silently assumed to match.
- [ ] Gaia references were proper-motion propagated to the SPHEREx observation epoch; the astrometric residuals were inspected by field and detector zone.
- [ ] PSF injection uses correct ePSF zone, oversampling and source sub-pixel phase.
- [ ] Completeness is reported with numerator/denominator and 95% Wilson intervals, flux bias and false-positive controls.
- [ ] The detection test is described accurately as a local matched-filter test, not a full blind catalogue benchmark.
- [ ] A separate holdout field/exposure set is preserved before tuning any thresholds.

A successful notebook execution can still produce an inconclusive scientific result. That is a valid outcome and should be recorded, not hidden.